In [9]:
from numpy.random import default_rng
import scanpy as sc
# import squidpy as sq
from anndata import AnnData
import scipy
from sklearn.metrics.cluster import adjusted_rand_score
import numpy as np
import pandas as pd
import seaborn as sns
import os
import torch
import pandas as pd
from sklearn import metrics
import multiprocessing as mp
from sklearn.metrics.cluster import normalized_mutual_info_score
import pickle
from sklearn.neighbors import kneighbors_graph
from sklearn.mixture import GaussianMixture
from sklearn.mixture import BayesianGaussianMixture
import time
from utils import cKD_refine_label
sc.logging.print_header()

/home/vanhoan310/miniconda3/envs/gnn/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


scanpy==1.9.3 anndata==0.9.2 umap==0.5.7 numpy==1.24.4 scipy==1.8.1 pandas==1.4.2 scikit-learn==1.3.2 statsmodels==0.14.1 python-igraph==0.11.8 pynndescent==0.5.13


In [10]:
from sklearn.cluster import (KMeans, MiniBatchKMeans, AgglomerativeClustering, DBSCAN, OPTICS, MeanShift, SpectralClustering, Birch)
from sklearn.mixture import GaussianMixture, BayesianGaussianMixture
from sklearn.metrics.cluster import adjusted_rand_score, normalized_mutual_info_score, homogeneity_score, completeness_score

In [11]:
## Parallel implementation
from joblib import Parallel, delayed
def process_slice_donor(batch_id, n_metacells):
    # print(sliceID, donorID)
    adata = adata_big[adata_big.obs['batch'] == batch_id].copy()
    if adata.shape[0] > 10:
        # n_clusters = 200 #100   
        # n_metacells = 200
        # Clustering
        connectivity = kneighbors_graph(adata.obsm['spatial'], n_neighbors=4, include_self=False)
        connectivity = 0.5 * (connectivity + connectivity.T)
        embedding = connectivity.dot(adata.obsm['X_pca'])

        clusteringobj = MiniBatchKMeans(n_clusters=n_metacells, random_state=42, tol=1e-3, max_iter=10000)
        adata.obs[GraphBGM] = clusteringobj.fit_predict(embedding)
        # adata.obs[GraphBGM] = refine_label(adata, radius=50, key=GraphBGM)
        adata.obs[GraphBGM] = cKD_refine_label(np.array(adata.obsm['spatial']), adata.obs[GraphBGM], k = 45)
        cluster_vec = np.unique(adata.obs[GraphBGM])
        clusterIDvec = []
        clusterValuevec = []
        cluster2Cellsvec = []
        for clusterID in cluster_vec:
        # for clusterID in range(n_clusters):
            sliceClusterID = batch_id + 'ClusterID' + str(clusterID)
            sliceClusterValue = np.mean(embedding[adata.obs[GraphBGM] == clusterID], axis=0)
            clusterIDvec.append(sliceClusterID)
            clusterValuevec.append(sliceClusterValue)
            cluster2Cellsvec.append(adata.obs_names[adata.obs[GraphBGM] == clusterID])
        return ((clusterIDvec, clusterValuevec, cluster2Cellsvec))

In [12]:
# MERFISH data
data_dir = '/data/hoan/spatial_transcriptomics/data/SDMBench/Data/378k/BrainAgingSpatialAtlas_MERFISH.h5ad'
adata_big=sc.read_h5ad(data_dir)
donor_id = np.unique(adata_big.obs['donor_id'])
adata_big.obsm['spatial'] = adata_big.obsm['spatial_coords']
adata_big.obs['batch'] = adata_big.obs['donor_id'].astype(str) +'Slice'+ adata_big.obs['slice'].astype(str)
adata_big.obs['Region'] = adata_big.obs['tissue']

In [13]:
GraphBGM = 'domain'#'GraphBGMv3a2full'
adata_big.obs['pseudoClustLabel'] = 0

In [14]:
# from utils import clustering_v3a2
from sklearn.cluster import (KMeans, MiniBatchKMeans, AgglomerativeClustering, SpectralClustering, Birch)
from sklearn.neighbors import kneighbors_graph
from GraphST.utils import refine_label

In [18]:
import numpy as np
metacell_numbers = [30, 40, 50, 80, 100, 150, 200]
for n_metacells in metacell_numbers:
    print('Number of metacells: ', n_metacells)
   
    # Parallel execution
    batch_vec = np.unique(adata_big.obs['batch'])
    # batch_vec = batch_vec[:5]
    results = Parallel(n_jobs=10)(delayed(process_slice_donor)(batch, n_metacells) for batch in batch_vec)
    sliceCluster2SampleID = {}
    pseudoSamples_column = []
    pseudoSamples_value = []
    for i in range(len(results)):
        if results[i] is not None:
            x, y, z = results[i]
            pseudoSamples_column += x
            pseudoSamples_value += y
            for j in range(len(x)):
                sliceCluster2SampleID[x[j]] = z[j]
    
    # pseudoSamples = pd.DataFrame(np.array(pseudoSamples_value)[:len(pseudoSamples_column),].T, columns = list(pseudoSamples_column))
    pseudoSamples = pd.DataFrame(np.array(pseudoSamples_value).T, columns = list(pseudoSamples_column))
    n_clusters = len(set(adata_big.obs['Region']))
    if np.nan in set(adata_big.obs['Region']):
        n_clusters = n_clusters-1 # remove nan
    
    # Define clustering algorithms
    clustering_algorithms = {
        'GraphBG': BayesianGaussianMixture(n_components=n_clusters, covariance_type='tied', random_state=42, init_params = 'random_from_data', n_init = 15, max_iter = 1000),
    }
    pseudoSamples_list = pseudoSamples_column
    if 'BrainAging' in pseudoSamples_list[0]:
        # print('Set batch for Brain Aging Dataset')
        batch_labels = [item[0:30] for item in pseudoSamples_list] # For 
    else:
        print('Set batch for Maynard dataset')
        batch_labels = [item[0:5] for item in pseudoSamples_list] # For
    # print(np.unique(batch_labels))
    X = pseudoSamples.T.values
    import scanpy.external as sce
    adata = sc.AnnData(X)
    adata.obsm['X_pca'] = X
    adata.obs["batch"] = batch_labels
    # time_st = time.time()
    # sc.pp.combat(adata, key="batch", inplace=True)
    # sce.pp.scanorama_integrate(adata, key="batch")
    sce.pp.harmony_integrate(adata, key="batch", max_iter_harmony=40)
    # time_ed = time.time()
    # time_combat = time_ed-time_st
    # adata
    X = adata.X
    if 'X_pca_harmony' in adata.obsm:
        print ("Use harmony")
        X  = adata.obsm['X_pca_harmony']
    # Store clustering results
    # time_st = time.time()
    results = {}
    for name, model in clustering_algorithms.items():
        if hasattr(model, 'fit_predict'):
            labels = model.fit_predict(X)
        else:
            model.fit(X)
            labels = model.predict(X)
        # Store labels for visualization
        results[name] = labels
        multi_clusterLabels = labels
        adata_big.obs['pseudoClustLabel'] = 0
        for i in range(len(pseudoSamples_list)):
            pseudoSamplesX = pseudoSamples_list[i]
            adata_big.obs.loc[sliceCluster2SampleID[pseudoSamplesX], 'pseudoClustLabel'] = multi_clusterLabels[i]
        #print(name, normalized_mutual_info_score(adata_big.obs['tissue'], adata_big.obs['pseudoClustLabel']))
        print('Spatial correction per slice')
        for batch in batch_vec:
            mask_ = (adata_big.obs['batch'] == batch)
            adata_big.obs.loc[mask_, 'pseudoClustLabel'] = cKD_refine_label(np.array(adata_big[mask_].obsm['spatial']),
                                                                            adata_big.obs.loc[mask_, 'pseudoClustLabel'], k = 45)
        true_labels = pd.factorize(adata_big.obs['Region'])[0]
    df_result = pd.DataFrame({"true_labels": true_labels[true_labels>0], "cluster_labels": adata_big.obs['pseudoClustLabel'][true_labels>0]})
    df_result.to_csv('output/GraphBG_sensitivity_n_metacells_'+str(n_metacells)+'.csv')
    print('============================================================================================')

Number of metacells:  50


2026-10-13 17:11:15,896 - harmonypy - INFO - Computing initial centroids with sklearn.KMeans...
2026-10-13 17:11:16,918 - harmonypy - INFO - sklearn.KMeans initialization complete.
2026-10-13 17:11:16,946 - harmonypy - INFO - Iteration 1 of 40
2026-10-13 17:11:17,435 - harmonypy - INFO - Iteration 2 of 40
2026-10-13 17:11:17,883 - harmonypy - INFO - Iteration 3 of 40
2026-10-13 17:11:18,335 - harmonypy - INFO - Iteration 4 of 40
2026-10-13 17:11:18,723 - harmonypy - INFO - Iteration 5 of 40
2026-10-13 17:11:19,050 - harmonypy - INFO - Iteration 6 of 40
2026-10-13 17:11:19,501 - harmonypy - INFO - Iteration 7 of 40
2026-10-13 17:11:19,712 - harmonypy - INFO - Iteration 8 of 40
2026-10-13 17:11:19,962 - harmonypy - INFO - Iteration 9 of 40
2026-10-13 17:11:20,357 - harmonypy - INFO - Iteration 10 of 40
2026-10-13 17:11:20,527 - harmonypy - INFO - Converged after 10 iterations


Use harmony
Spatial correction per slice


In [19]:
### Data analysis
from sklearn.metrics.cluster import  normalized_mutual_info_score, homogeneity_score, completeness_score

In [24]:
import numpy as np
import pandas as pd
metacell_numbers = [30, 40, 50, 80, 100, 150, 200]
for n_metacells in metacell_numbers:
    print('Number of metacells: ', n_metacells)
    df_result = pd.read_csv('output/GraphBG_sensitivity_n_metacells_'+str(n_metacells)+'.csv')
    true_labels = df_result['true_labels']
    cluster_labels = df_result['cluster_labels']
    print(normalized_mutual_info_score(true_labels, cluster_labels))
    print('============================================================================================')

Number of metacells:  30
0.6637724352286555
Number of metacells:  40
0.6826161498008506
Number of metacells:  50
0.7091723587163543
Number of metacells:  80
0.7070074309096528
Number of metacells:  100
0.6836028925967776
Number of metacells:  150
0.7194705087946788
Number of metacells:  200
0.7089705045586683
